In [ ]:
# env) medeagent-kcd
# file) medagent-basics/hf-agents_v01.ipynb

from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:3b", temperature=0)

In [2]:
# test
response = llm.invoke("Hi! please introduce yourself.")
print(response.content)

Nice to meet you! I'm an artificial intelligence model, which means I'm a computer program designed to simulate human-like conversations and answer questions to the best of my ability. I don't have a personal name, but I'm often referred to as a "chatbot" or a "language model."

I was created to assist and provide information on a wide range of topics, from science and history to entertainment and culture. I'm constantly learning and improving my knowledge base, so I can provide more accurate and helpful responses over time.

I don't have personal experiences, emotions, or opinions like humans do, but I'm here to help and provide information to the best of my ability. How can I assist you today?


Format

In [3]:
from pydantic import BaseModel
from typing import Literal

# input format 
class PatientState(BaseModel):
    patient_id: str
    age: int
    sex: str
    sbp: float
    hr: float
    creatinine: float
    bun: float
    sodium: float
    potassium: float
    diuretic_name: str
    diuretic_dose_mg: float
    urine_6h_ml: float

# output format
class SpecialistResult(BaseModel):
    agent: Literal["cardiology", "nephrology", "pharmacy"]
    assessment: str
    concern_level: Literal["low", "moderate", "high"]
    confidence: Literal["low", "moderate", "high"]
    needs_review: bool
    reason: str

Mock patient

In [4]:
patient = PatientState(
    patient_id="P001",
    age=72,
    sex="F",
    sbp=102,
    hr=96,
    creatinine=1.8,
    bun=35,
    sodium=132,
    potassium=4.7,
    diuretic_name="furosemide",
    diuretic_dose_mg=40,
    urine_6h_ml=250
)
patient

PatientState(patient_id='P001', age=72, sex='F', sbp=102.0, hr=96.0, creatinine=1.8, bun=35.0, sodium=132.0, potassium=4.7, diuretic_name='furosemide', diuretic_dose_mg=40.0, urine_6h_ml=250.0)

Cardiology Agent prompt

In [5]:
from langchain_core.prompts import ChatPromptTemplate

cardiology_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are the Cardiology Agent of HF-PathGuard.
        Your role is to assess early diuretic response
        in a heart-failure ICU patient.
        Focus on:
        - urine output
        - blood pressure
        - heart rate
        - overall early diuretic response

        Do not make a definitive diagnosis.
        Do not provide direct treatment orders.
        If the information is insufficient or concerning,
        recommend clinician review.
        """
    ),
    (
        "human",
        """
        Patient information:

        Age: {age}
        Sex: {sex}
        SBP: {sbp}
        HR: {hr}
        Creatinine: {creatinine}
        BUN: {bun}
        Sodium: {sodium}
        Potassium: {potassium}
        Diuretic: {diuretic_name}
        Dose: {diuretic_dose_mg} mg
        Urine output over 6 hours:
        {urine_6h_ml} mL

        Please assess the patient's early diuretic response.
        """
    )
])

In [6]:
# structured output + LCEL
cardiology_llm = llm.with_structured_output(SpecialistResult)
cardiology_chain = (cardiology_prompt | cardiology_llm)

In [7]:
cardiology_result = cardiology_chain.invoke(patient.model_dump())
cardiology_result

SpecialistResult(agent='cardiology', assessment='early diuretic response', concern_level='low', confidence='moderate', needs_review=True, reason="Insufficient data on urine output over a longer period and no information on the patient's symptoms or clinical status. However, the patient's urine output is 250.0 mL over 6 hours, which is relatively low considering the dose of furosemide administered. The patient's creatinine and BUN levels are elevated, indicating possible renal impairment. The patient's blood pressure is 102.0 mmHg, which is within a relatively normal range. The patient's heart rate is 96.0 bpm, which is slightly elevated. The patient's potassium level is 4.7 mmol/L, which is within a relatively normal range. The patient's sodium level is 132.0 mmol/L, which is slightly low. Overall, the patient's early diuretic response appears to be limited, and further assessment is needed to determine the effectiveness of the diuretic therapy. Clinician review is recommended to evalu

Nephrology Agent Prompt

In [8]:
nephrology_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are the Nephrology Agent of HF-PathGuard.
        Your role is to assess renal and electrolyte safety
        after loop-diuretic administration.
        Focus on:
        - creatinine
        - BUN
        - sodium
        - potassium
        - possible renal/electrolyte concern

        Do not make a definitive diagnosis.
        Do not provide direct treatment orders.
        If information is insufficient or concerning,
        recommend clinician review.
        """
    ),
    (
        "human",
        """
        Patient information:
        Age: {age}
        Sex: {sex}
        SBP: {sbp}
        HR: {hr}
        Creatinine: {creatinine}
        BUN: {bun}
        Sodium: {sodium}
        Potassium: {potassium}
        Diuretic: {diuretic_name}
        Dose: {diuretic_dose_mg} mg
        Urine output over 6 hours: {urine_6h_ml} mL

        Please assess renal and electrolyte safety.
        """
    )
])

In [9]:
nephrology_chain = (nephrology_prompt | llm.with_structured_output(SpecialistResult))
nephrology_result = nephrology_chain.invoke(patient.model_dump())
nephrology_result

SpecialistResult(agent='nephrology', assessment='renal and electrolyte safety', concern_level='moderate', confidence='low', needs_review=True, reason="Patient has elevated creatinine (1.8) and BUN (35.0), indicating possible renal impairment. The furosemide dose is moderate (40.0 mg), and the urine output is relatively low (250.0 mL over 6 hours). The patient's potassium level is elevated (4.7), which may be related to the diuretic use. The sodium level is slightly low (132.0), but this may be a normal response to diuretic use. Overall, the patient's renal and electrolyte safety is a concern, and further review is recommended.")

Pharmacy Agent

In [10]:
pharmacy_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are the Pharmacy Agent of HF-PathGuard.
        Your role is to review loop-diuretic exposure
        and medication-related safety concerns.
        Focus on:
        - diuretic name
        - diuretic dose
        - blood pressure
        - renal function
        - electrolyte values

        Do not make a definitive diagnosis.
        Do not provide direct treatment orders.
        If medication-related information is insufficient
        or potentially unsafe, recommend clinician/pharmacist review.
        """
    ),
    (
        "human",
        """
        Patient information:
        Age: {age}
        Sex: {sex}
        SBP: {sbp}
        HR: {hr}
        Creatinine: {creatinine}
        BUN: {bun}
        Sodium: {sodium}
        Potassium: {potassium}
        Diuretic: {diuretic_name}
        Dose: {diuretic_dose_mg} mg
        Urine output over 6 hours: {urine_6h_ml} mL

        Please assess medication-related safety.
        """
    )
])

In [11]:
pharmacy_chain = (pharmacy_prompt | llm.with_structured_output(SpecialistResult)
)
pharmacy_result = pharmacy_chain.invoke(patient.model_dump())
pharmacy_result

SpecialistResult(agent='pharmacy', assessment='Medication-related safety assessment for furosemide in a 72-year-old female patient.', concern_level='moderate', confidence='low', needs_review=True, reason="The patient's serum creatinine level is elevated at 1.8, which may indicate impaired renal function. Furosemide is a loop diuretic that can exacerbate renal impairment. Additionally, the patient's potassium level is low at 4.7, which may be a concern given the diuretic's potential to increase potassium excretion. The patient's urine output is 250 mL over 6 hours, which is relatively low. Further evaluation is needed to determine the safety of this medication in this patient.")

In [12]:
print(cardiology_result)
print(nephrology_result)
print(pharmacy_result)

agent='cardiology' assessment='early diuretic response' concern_level='low' confidence='moderate' needs_review=True reason="Insufficient data on urine output over a longer period and no information on the patient's symptoms or clinical status. However, the patient's urine output is 250.0 mL over 6 hours, which is relatively low considering the dose of furosemide administered. The patient's creatinine and BUN levels are elevated, indicating possible renal impairment. The patient's blood pressure is 102.0 mmHg, which is within a relatively normal range. The patient's heart rate is 96.0 bpm, which is slightly elevated. The patient's potassium level is 4.7 mmol/L, which is within a relatively normal range. The patient's sodium level is 132.0 mmol/L, which is slightly low. Overall, the patient's early diuretic response appears to be limited, and further assessment is needed to determine the effectiveness of the diuretic therapy. Clinician review is recommended to evaluate the patient's resp

Deterministic Rule (w/o Chief Agent)

In [13]:
review_reasons = []

for result in [cardiology_result, nephrology_result, pharmacy_result]:
    if result.needs_review:
        review_reasons.append(
            f"{result.agent}: {result.reason}"
        )

overall_status = ("REVIEW"
    if len(review_reasons) > 0
    else "OK"
)
# if len(review_reasons) > 0:
#     overall_status = "REVIEW"
# else:
#     overall_status = "OK"

In [14]:
# check shadow
# SafetyFlag(
#     flag_type="agent_disagreement",
#     severity="moderate",
#     message="Potassium 4.7 was interpreted differently by nephrology and pharmacy agents."
# )

SafetyFlag

In [15]:
from pydantic import BaseModel
from typing import Literal # 허용할 문자열의 정확한 값 지정

class SafetyFlag(BaseModel):
    # 무슨 문제인지
    flag_type: Literal[
        "lab_conflict",
        "agent_disagreement",
        "missing_data",
        "low_confidence"
    ]
    severity: Literal["low", "moderate", "high"] # 중요도
    message: str # 아유

In [16]:
SafetyFlag(
    flag_type="agent_disagreement",
    severity="moderate",
    message="Potassium 4.7 was interpreted differently by nephrology and pharmacy agents."
)

SafetyFlag(flag_type='agent_disagreement', severity='moderate', message='Potassium 4.7 was interpreted differently by nephrology and pharmacy agents.')

Deterministic Safety Rule (LLM X)

In [17]:
safety_flags = []

if 3.5 <= patient.potassium <= 5.0:
    potassium_status = "normal"
elif patient.potassium < 3.5:
    potassium_status = "low"
else:
    potassium_status = "high"

In [18]:
pharmacy_text = pharmacy_result.reason.lower() # 소문자 변환
nephrology_text = nephrology_result.reason.lower() 

if potassium_status == "normal":
    if "potassium level is low" in pharmacy_text:
        safety_flags.append(
            SafetyFlag(
                flag_type="lab_conflict",
                severity="high",
                message="Pharmacy agent described potassium 4.7 as low, but deterministic rule classified it as normal."
            )
        )

    if "potassium level is elevated" in nephrology_text:
        safety_flags.append(
            SafetyFlag(
                flag_type="lab_conflict",
                severity="high",
                message="Nephrology agent described potassium 4.7 as elevated, but deterministic rule classified it as normal."
            )
        )

In [19]:
safety_flags

[SafetyFlag(flag_type='lab_conflict', severity='high', message='Pharmacy agent described potassium 4.7 as low, but deterministic rule classified it as normal.'),
 SafetyFlag(flag_type='lab_conflict', severity='high', message='Nephrology agent described potassium 4.7 as elevated, but deterministic rule classified it as normal.')]

Agent disagreement

In [20]:
# 차이가 크면 flag 추가

concern_score = {
    "low": 0,
    "moderate": 1,
    "high": 2
}

agent_results = [
    cardiology_result,
    nephrology_result,
    pharmacy_result
]

scores = [
    concern_score[result.concern_level]
    for result in agent_results
]

if max(scores) - min(scores) >= 2:
    safety_flags.append(
        SafetyFlag(
            flag_type="agent_disagreement",
            severity="moderate",
            message="Specialist agents show substantial disagreement in concern level."
        )
    )

In [21]:
for result in agent_results:
    if result.confidence == "low":
        safety_flags.append(
            SafetyFlag(
                flag_type="low_confidence",
                severity="moderate",
                message=f"{result.agent} agent reported low confidence."
            )
        )

In [22]:
safety_flags

[SafetyFlag(flag_type='lab_conflict', severity='high', message='Pharmacy agent described potassium 4.7 as low, but deterministic rule classified it as normal.'),
 SafetyFlag(flag_type='lab_conflict', severity='high', message='Nephrology agent described potassium 4.7 as elevated, but deterministic rule classified it as normal.'),
 SafetyFlag(flag_type='low_confidence', severity='moderate', message='nephrology agent reported low confidence.'),
 SafetyFlag(flag_type='low_confidence', severity='moderate', message='pharmacy agent reported low confidence.')]

Final Report Scheme

In [23]:
class FinalReport(BaseModel):
    patient_id: str
    cardiology: SpecialistResult
    nephrology: SpecialistResult
    pharmacy: SpecialistResult
    safety_flags: list[SafetyFlag] # 추가
    overall_status: Literal["OK", "REVIEW"]
    review_reasons: list[str]

Review region update

In [24]:
review_reasons = []

for result in [
    cardiology_result,
    nephrology_result,
    pharmacy_result
]:
    if result.needs_review:
        review_reasons.append(
            f"{result.agent}: {result.reason}"
        )

for flag in safety_flags:
    review_reasons.append(
        f"safety: {flag.message}"
    )

In [25]:
if len(review_reasons) > 0:
    overall_status = "REVIEW"
else:
    overall_status = "OK"

Final report update

In [26]:
class FinalReport(BaseModel):
    patient_id: str
    cardiology: SpecialistResult
    nephrology: SpecialistResult
    pharmacy: SpecialistResult

    safety_flags: list[SafetyFlag]

    overall_status: Literal["OK", "REVIEW"]
    review_reasons: list[str]

In [27]:
final_report = FinalReport(
    patient_id=patient.patient_id,
    cardiology=cardiology_result,
    nephrology=nephrology_result,
    pharmacy=pharmacy_result,
    safety_flags=safety_flags,
    overall_status=overall_status,
    review_reasons=review_reasons
)
final_report

FinalReport(patient_id='P001', cardiology=SpecialistResult(agent='cardiology', assessment='early diuretic response', concern_level='low', confidence='moderate', needs_review=True, reason="Insufficient data on urine output over a longer period and no information on the patient's symptoms or clinical status. However, the patient's urine output is 250.0 mL over 6 hours, which is relatively low considering the dose of furosemide administered. The patient's creatinine and BUN levels are elevated, indicating possible renal impairment. The patient's blood pressure is 102.0 mmHg, which is within a relatively normal range. The patient's heart rate is 96.0 bpm, which is slightly elevated. The patient's potassium level is 4.7 mmol/L, which is within a relatively normal range. The patient's sodium level is 132.0 mmol/L, which is slightly low. Overall, the patient's early diuretic response appears to be limited, and further assessment is needed to determine the effectiveness of the diuretic therapy